# 00 — Data acquisition

Cell-by-cell view of three scripts:

- `code/preprocessing/stac_manifest.py`
- `code/preprocessing/fetch_subsets.py`
- `code/preprocessing/check_coverage.py`

This notebook resolves the STAC items, builds the common `EPSG:32646` 10 m grid,
fetches windowed AOI subsets, and verifies coverage. It needs network access to
the Planetary Computer. The `.py` scripts remain the canonical CLI.

In [1]:
# Setup — run this cell first.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "code" / "config" / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code" / "preprocessing"))
print("project root:", ROOT)

project root: D:\DIVP PROJECT


## Step 1 — Resolve STAC items and write the metadata manifest

Mirrors `stac_manifest.py`. No pixels are downloaded in this step.

In [2]:
from common import (
    get_item, load_config, rel_path, save_json, stac_client, utc_now,
)

PROPERTY_KEYS = {
    "datetime", "platform", "constellation", "instruments",
    "sar:instrument_mode", "sar:frequency_band", "sar:polarizations",
    "sar:product_type", "sat:orbit_state", "sat:relative_orbit",
    "s2:mgrs_tile", "eo:cloud_cover", "proj:epsg", "proj:shape", "gsd",
}


def item_record(item):
    properties = {k: item.properties.get(k) for k in PROPERTY_KEYS if k in item.properties}
    assets = {}
    for name, asset in item.assets.items():
        assets[name] = {
            "href": asset.href,
            "media_type": asset.media_type,
            "roles": asset.roles,
            "title": asset.title,
        }
    return {
        "collection": item.collection_id,
        "item_id": item.id,
        "datetime": item.datetime.isoformat() if item.datetime else None,
        "bbox": item.bbox,
        "properties": properties,
        "assets": assets,
    }


def resolve(client, collection, item_id):
    item = get_item(client, collection, item_id)
    print(f"  OK  {collection} / {item_id}")
    return item_record(item)

In [3]:
cfg = load_config()
client = stac_client(cfg)
collections = cfg["stac"]["collections"]
items = cfg["items"]

datasets = {}
print("Resolving STAC items:")
datasets["s1_rtc_pre"] = resolve(client, collections["s1_rtc"], items["s1_rtc_pre"])
datasets["s1_rtc_post"] = resolve(client, collections["s1_rtc"], items["s1_rtc_post"])
for index, tile_id in enumerate(items["s2_post_tiles"]):
    datasets[f"s2_post_tile_{index + 1}"] = resolve(client, collections["s2_l2a"], tile_id)
datasets["dem"] = resolve(client, collections["dem"], items["dem"])
datasets["jrc_water"] = resolve(client, collections["jrc_water"], items["jrc_water"])
datasets["worldcover"] = resolve(client, collections["worldcover"], items["worldcover"])
print(f"resolved {len(datasets)} dataset entries")

Resolving STAC items:
  OK  sentinel-1-rtc / S1A_IW_GRDH_1SDV_20230611T115714_20230611T115739_048938_05E286_rtc
  OK  sentinel-1-rtc / S1A_IW_GRDH_1SDV_20230623T115715_20230623T115740_049113_05E7DB_rtc
  OK  sentinel-2-l2a / S2B_MSIL2A_20230625T042709_R133_T46RCQ_20230625T090934
  OK  sentinel-2-l2a / S2B_MSIL2A_20230625T042709_R133_T46RCP_20230625T092053
  OK  cop-dem-glo-30 / Copernicus_DSM_COG_10_N26_00_E091_00_DEM
  OK  jrc-gsw / 90E_30Nv1_3_2020
  OK  esa-worldcover / ESA_WorldCover_10m_2021_v200_N24E090
resolved 7 dataset entries


In [4]:
manifest = {
    "generated_utc": utc_now(),
    "status": "metadata-only",
    "note": (
        "Asset hrefs are unsigned and expire after signing. "
        "The fetch step re-resolves each item and signs it at fetch time."
    ),
    "stac_api": cfg["stac"]["api"],
    "aoi": {"name": cfg["aoi"]["name"], "bbox_wgs84": cfg["aoi"]["bbox"]},
    "grid": {"crs": cfg["grid"]["crs"], "resolution_m": cfg["grid"]["resolution_m"]},
    "event": cfg["event"],
    "collections": collections,
    "assets": cfg["stac"]["assets"],
    "datasets": datasets,
    "local_files": {},
}
manifest_path = rel_path(cfg, "manifest")
save_json(manifest_path, manifest)
print("wrote", manifest_path)

wrote D:\DIVP PROJECT\data\manifest.json


## Step 2 — Build the common grid and fetch Sentinel-1 RTC

Mirrors the first half of `fetch_subsets.py`.

In [5]:
from common import (
    combine_mosaics, load_json, read_reprojected, sha256_file, sign_item,
    target_grid, write_geotiff,
)
from rasterio.enums import Resampling
import numpy as np
import matplotlib.pyplot as plt

In [6]:
manifest = load_json(manifest_path)
dst_transform, width, height, dst_crs = target_grid(cfg)
manifest["grid"].update({
    "width": width,
    "height": height,
    "transform": list(dst_transform)[:6],
    "pixel_area_m2": float(cfg["grid"]["resolution_m"]) ** 2,
})
print(f"grid: {width} x {height} @ {cfg['grid']['resolution_m']} m, CRS {dst_crs}")

grid: 1713 x 1346 @ 10 m, CRS EPSG:32646


In [7]:
def register(manifest, key, path, root):
    manifest["local_files"][key] = {
        "path": str(path.relative_to(root)).replace("\\", "/"),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }
    print(f"  wrote {path.relative_to(root)} ({path.stat().st_size / 1e6:.2f} MB)")


def fetch_s1(cfg, client, manifest, dst_crs, dst_transform, width, height, root):
    out_dir = rel_path(cfg, "data_raw") / "sentinel1"
    jobs = [
        ("s1_rtc_pre_vv", manifest["datasets"]["s1_rtc_pre"]["item_id"], "vv"),
        ("s1_rtc_pre_vh", manifest["datasets"]["s1_rtc_pre"]["item_id"], "vh"),
        ("s1_rtc_post_vv", manifest["datasets"]["s1_rtc_post"]["item_id"], "vv"),
        ("s1_rtc_post_vh", manifest["datasets"]["s1_rtc_post"]["item_id"], "vh"),
    ]
    cache = {}
    for key, item_id, pol in jobs:
        if item_id not in cache:
            cache[item_id] = sign_item(
                get_item(client, cfg["stac"]["collections"]["s1_rtc"], item_id)
            )
        item = cache[item_id]
        arr = read_reprojected(
            item.assets[pol].href, dst_crs, dst_transform, width, height,
            Resampling.bilinear, dst_nodata=-32768.0,
        )
        if arr is None:
            raise RuntimeError(f"No overlap for {key}")
        path = out_dir / f"{key}.tif"
        write_geotiff(path, arr, dst_transform, dst_crs, -32768.0)
        register(manifest, key, path, root)

In [8]:
root = Path(cfg["_root"])
print("Fetching S1 RTC subsets...")
fetch_s1(cfg, client, manifest, dst_crs, dst_transform, width, height, root)

Fetching S1 RTC subsets...
  wrote data\raw\sentinel1\s1_rtc_pre_vv.tif (8.29 MB)
  wrote data\raw\sentinel1\s1_rtc_pre_vh.tif (8.22 MB)
  wrote data\raw\sentinel1\s1_rtc_post_vv.tif (8.29 MB)
  wrote data\raw\sentinel1\s1_rtc_post_vh.tif (8.22 MB)


## Step 3 — Fetch Sentinel-2, DEM, JRC water, WorldCover

In [9]:
def fetch_s2(cfg, client, manifest, dst_crs, dst_transform, width, height, root):
    out_dir = rel_path(cfg, "data_raw") / "sentinel2"
    tile_items = [
        manifest["datasets"][e]["item_id"]
        for e in manifest["datasets"]
        if e.startswith("s2_post_tile_")
    ]
    signed = [
        sign_item(get_item(client, cfg["stac"]["collections"]["s2_l2a"], item_id))
        for item_id in tile_items
    ]
    band_resampling = {
        "B03": Resampling.bilinear,
        "B11": Resampling.bilinear,
        "SCL": Resampling.nearest,
    }
    for band, resampling in band_resampling.items():
        arrays = []
        for item in signed:
            arr = read_reprojected(
                item.assets[band].href, dst_crs, dst_transform, width, height,
                resampling, dst_nodata=0,
            )
            if arr is not None:
                arrays.append(arr)
        if not arrays:
            raise RuntimeError(f"No S2 overlap for {band}")
        mosaic = combine_mosaics(arrays, nodata=0)
        key = f"s2_post_{band}"
        path = out_dir / f"{key}.tif"
        write_geotiff(path, mosaic, dst_transform, dst_crs, 0)
        register(manifest, key, path, root)


def fetch_single(cfg, client, manifest, key, subdir, collection, item_id, asset,
                 resampling, dst_crs, dst_transform, width, height, root, nodata):
    item = sign_item(get_item(client, collection, item_id))
    arr = read_reprojected(
        item.assets[asset].href, dst_crs, dst_transform, width, height,
        resampling, dst_nodata=nodata,
    )
    if arr is None:
        raise RuntimeError(f"No overlap for {key}")
    path = rel_path(cfg, "data_raw") / subdir / f"{key}_aoi.tif"
    write_geotiff(path, arr, dst_transform, dst_crs, nodata)
    register(manifest, key, path, root)

In [10]:
print("Fetching S2 subsets (mosaicking tiles)...")
fetch_s2(cfg, client, manifest, dst_crs, dst_transform, width, height, root)

print("Fetching DEM...")
fetch_single(
    cfg, client, manifest, "dem", "dem", cfg["stac"]["collections"]["dem"],
    manifest["datasets"]["dem"]["item_id"], cfg["stac"]["assets"]["dem"],
    Resampling.bilinear, dst_crs, dst_transform, width, height, root, -9999.0,
)

print("Fetching JRC occurrence...")
fetch_single(
    cfg, client, manifest, "jrc_occurrence", "reference",
    cfg["stac"]["collections"]["jrc_water"],
    manifest["datasets"]["jrc_water"]["item_id"], cfg["stac"]["assets"]["jrc_water"],
    Resampling.nearest, dst_crs, dst_transform, width, height, root, 255,
)

print("Fetching WorldCover...")
fetch_single(
    cfg, client, manifest, "worldcover", "landcover",
    cfg["stac"]["collections"]["worldcover"],
    manifest["datasets"]["worldcover"]["item_id"], cfg["stac"]["assets"]["worldcover"],
    Resampling.nearest, dst_crs, dst_transform, width, height, root, 0,
)

Fetching S2 subsets (mosaicking tiles)...
  wrote data\raw\sentinel2\s2_post_B03.tif (3.27 MB)
  wrote data\raw\sentinel2\s2_post_B11.tif (3.51 MB)
  wrote data\raw\sentinel2\s2_post_SCL.tif (0.09 MB)
Fetching DEM...
  wrote data\raw\dem\dem_aoi.tif (6.91 MB)
Fetching JRC occurrence...
  wrote data\raw\reference\jrc_occurrence_aoi.tif (0.05 MB)
Fetching WorldCover...
  wrote data\raw\landcover\worldcover_aoi.tif (0.19 MB)


## Step 4 — Quicklook figure

In [11]:
def quicklook(cfg, manifest, root):
    import rasterio
    base = rel_path(cfg, "data_raw") / "sentinel1"
    pre = rasterio.open(base / "s1_rtc_pre_vv.tif").read(1).astype("float32")
    post = rasterio.open(base / "s1_rtc_post_vv.tif").read(1).astype("float32")

    def to_db(arr):
        out = np.full(arr.shape, np.nan, dtype="float32")
        valid = (arr > 0) & (arr != -32768.0)
        out[valid] = 10.0 * np.log10(arr[valid])
        return out

    pre_db, post_db = to_db(pre), to_db(post)
    diff = post_db - pre_db

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, data, title, kw in (
        (axes[0], pre_db, "Pre-event VV (dB)", {"vmin": -25, "vmax": 5}),
        (axes[1], post_db, "Post-event VV (dB)", {"vmin": -25, "vmax": 5}),
        (axes[2], diff, "Post - pre (dB)", {"vmin": -10, "vmax": 10}),
    ):
        im = ax.imshow(data, cmap="gray" if "diff" not in title else "RdBu", **kw)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle("Day 2 quicklook — Guwahati AOI (EPSG:32646, 10 m)")
    fig.tight_layout()

    out = rel_path(cfg, "outputs") / "figures" / "day2_quicklook.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=150)
    plt.close(fig)
    register(manifest, "day2_quicklook", out, root)


quicklook(cfg, manifest, root)

  wrote outputs\figures\day2_quicklook.png (1.19 MB)


## Step 5 — Save the manifest and verify coverage

In [12]:
manifest["status"] = "subsets-fetched"
manifest["fetched_utc"] = utc_now()
save_json(manifest_path, manifest)
print("updated", manifest_path)

updated D:\DIVP PROJECT\data\manifest.json


In [13]:
import csv
import rasterio

expected_crs = cfg["grid"]["crs"]
expected_res = float(cfg["grid"]["resolution_m"])

rows = []
warnings = []
for key, meta in manifest["local_files"].items():
    path = root / meta["path"]
    if path.suffix.lower() not in (".tif", ".tiff"):
        continue
    with rasterio.open(path) as src:
        data = src.read(1, out_shape=(1, min(512, src.height), min(512, src.width)))
        nodata = src.nodata
        valid = np.ones(data.shape, dtype=bool) if nodata is None else data != nodata
        valid_fraction = float(valid.mean())
        rows.append({
            "key": key, "crs": str(src.crs),
            "res_x": f"{abs(src.transform.a):.3f}",
            "res_y": f"{abs(src.transform.e):.3f}",
            "width": src.width, "height": src.height,
            "nodata": nodata,
            "valid_fraction": f"{valid_fraction:.4f}",
            "bounds": str(tuple(round(v, 2) for v in src.bounds)),
        })
        if str(src.crs) != expected_crs:
            warnings.append(f"{key}: CRS {src.crs} != {expected_crs}")
        if abs(abs(src.transform.a) - expected_res) > 0.01:
            warnings.append(f"{key}: res {src.transform.a} != {expected_res}")
        if valid_fraction < 0.95:
            warnings.append(f"{key}: valid fraction {valid_fraction:.3f} < 0.95")

out = rel_path(cfg, "outputs") / "tables" / "day2_coverage.csv"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

grid = manifest["grid"]
area_km2 = grid["width"] * grid["height"] * grid["pixel_area_m2"] / 1e6
print(f"Grid: {grid['width']} x {grid['height']} px at "
      f"{cfg['grid']['resolution_m']} m = {area_km2:.2f} km^2")
print("wrote", out)
print("WARNINGS:", warnings if warnings else "none")

Grid: 1713 x 1346 px at 10 m = 230.57 km^2
wrote D:\DIVP PROJECT\outputs\tables\day2_coverage.csv
WARNINGS: none
